# Day 56: t-SNE for Visualization

## What is t-SNE?
t-SNE (t-distributed Stochastic Neighbor Embedding) is a dimensionality reduction technique designed specifically for visualization rather than general-purpose compression. Where PCA finds linear axes of maximum variance, t-SNE works by modeling how likely each point is to be a "neighbor" of every other point in the original high-dimensional space, then arranging points in 2D/3D so that those neighborhood probabilities are preserved as closely as possible — points that were close in high-dimensional space stay close in the plot, while distances between far-apart clusters are not meaningfully preserved. This makes it very good at revealing cluster structure visually, even when that structure is non-linear and PCA would blur it together. We'll use the digits dataset (8x8 pixel images of handwritten digits, flattened to 64 features) to see this in action — `X.shape` below confirms we have 64 features per sample.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X, y = digits.data, digits.target
print(X.shape)  # (n_samples, 64) -- each 8x8 digit image flattened into 64 pixel-intensity features

## Applying t-SNE
Because t-SNE optimizes a non-linear objective (matching neighbor probability distributions between high- and low-dimensional space) rather than a closed-form linear projection like PCA, it usually separates visually distinct groups much more crisply — but that comes at the cost of being much slower to compute and having no simple, reusable transformation you can apply to new data. `perplexity` (here 30) is t-SNE's most important tuning parameter: it roughly controls how many neighbors each point considers when the algorithm decides what "local structure" to preserve, balancing attention between very local detail (low perplexity) and broader structure (high perplexity) — typical values are between 5 and 50. `random_state=42` matters more here than with PCA because t-SNE's optimization involves randomness (it starts from a random initial layout and uses gradient descent), so results can look different across runs otherwise.

In [ ]:
from sklearn.manifold import TSNE

# perplexity roughly sets how many neighbors each point "pays attention to" when
# preserving local structure; random_state fixes the random initialization/optimization
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X)

## Visualizing Digit Clusters
Now we plot the 2D t-SNE coordinates, coloring each point by its true digit label (0-9). Because t-SNE preserves local neighborhoods, images of the same digit — which look similar in pixel space — should land near each other, forming visually distinct, separated blobs even though t-SNE was never told the labels (it's unsupervised, just like PCA). When you look at the plot, focus on whether same-colored points cluster together and how cleanly separated the clusters are from each other; don't read anything into the absolute distances between clusters or their overall arrangement, since t-SNE only tries to preserve local closeness, not global geometry.

In [ ]:
import matplotlib.pyplot as plt

# Color by true digit label to visually verify t-SNE separated the 10 digit classes
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', s=10)
plt.legend(*scatter.legend_elements(), title='Digit')
plt.title('t-SNE Visualization of Handwritten Digits')
plt.show()

## PCA vs t-SNE
These two techniques solve different problems and shouldn't be used interchangeably. PCA is a fast, linear, deterministic transformation that preserves global variance structure, has an explicit formula you can apply to new/unseen data, and is safe to use as a preprocessing step before feeding data into another model (e.g. reducing 100 features to 20 before training a classifier). t-SNE is slow, non-linear, and stochastic (results vary by random seed and even by perplexity), and it has no simple way to transform new points after fitting — every run essentially re-solves the layout from scratch. Because t-SNE distorts global distances and cluster sizes on purpose to emphasize local neighbor structure, its output coordinates carry no meaningful scale or global relationship, so you should treat it strictly as a visualization tool and never feed its output into a downstream model.

In [ ]:
# Quick recap of when to reach for each technique
print('PCA -> linear, fast, keeps global structure, usable as model input')
print('t-SNE -> non-linear, slow, keeps local structure, visualization only')